(ต่อ) จาก cleansing data เสร็จ

### STEP 6: Load cleaned data

In [5]:
import pandas as pd
from tqdm import tqdm  # เปลี่ยนจาก tqdm.notebook เป็น tqdm ธรรมดา

with tqdm(total=1, desc="Loading Data") as pbar:
    df = pd.read_parquet("../data/processed/accidents_clean.parquet")
    pbar.update(1)





Loading Data: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]


### STEP 7: Fix Time Missing (เชิง modeling)
* ตอนนี้เราจะ ไม่ impute เวลา
* แต่สร้าง indicator แทน (สำคัญมาก)

ดีกว่า fill ค่า fake เพราะ:
* model เรียนรู้ pattern ได้เอง
* ไม่โกหกข้อมูล

In [6]:
df['Has_End_Time'] = df['End_Time'].notna().astype(int)
df['Has_End_Location'] = df['End_Lat'].notna().astype(int)


### STEP 8: Transform Distance (heavy skew)
* Distance เป็น long-tail → ใช้ log

In [7]:
df['Log_Distance'] = np.log1p(df['Distance(mi)'])


### STEP 9: Transform Duration (เฉพาะที่ valid)
* NaN จะยังเป็น NaN → ถูกต้อง

In [8]:
df['Log_Duration'] = np.log1p(
    df['Accident_Duration_Min'].clip(lower=0)
)


### STEP 10: Encode Cyclic Time

In [9]:
df['Hour_sin'] = np.sin(2 * np.pi * df['Start_Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Start_Hour'] / 24)

df['Month_sin'] = np.sin(2 * np.pi * df['Start_Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Start_Month'] / 12)


### STEP 11: Reduce high-cardinality categorical

In [10]:
top_states = df['State'].value_counts().nlargest(10).index
df['State_Grouped'] = np.where(
    df['State'].isin(top_states),
    df['State'],
    'Other'
)


### STEP 12: Save feature-ready data

In [11]:
df.to_parquet(
    "../data/processed/accidents_features.parquet",
    index=False
)
